# llama.cpp RPC Worker - Colab Setup

**FIRST:** Runtime → Change runtime type → GPU → Save

**THEN:** Run cells in order

In [ ]:
import subprocess

try:
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True, check=True
    )
    print(f"✅ GPU: {r.stdout.strip()}")
except Exception:
    print("❌ NO GPU — Runtime → Change runtime type → T4 GPU → Save")
    raise SystemExit(1)

✅ GPU: Tesla T4, 15360 MiB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted — binaries will be cached here for fast restarts")

Mounted at /content/drive
✅ Drive mounted — binaries will be cached here for fast restarts


In [ ]:
import subprocess, os

GO_VERSION = "1.22.4"

if not os.path.exists('/usr/local/go/bin/go'):
    print(f"📦 Downloading Go {GO_VERSION}…")
    subprocess.run(
        ['wget', '-q', '-O', '/tmp/go.tar.gz',
         f'https://go.dev/dl/go{GO_VERSION}.linux-amd64.tar.gz'],
        check=True
    )
    subprocess.run(['sudo', 'tar', '-C', '/usr/local', '-xzf', '/tmp/go.tar.gz'], check=True)
    os.remove('/tmp/go.tar.gz')
    print(f"✅ Go {GO_VERSION} installed")
else:
    print("✅ Go already installed — skipping download")

os.environ['PATH']   = '/usr/local/go/bin:' + os.environ.get('PATH', '')
os.environ['GOPATH'] = '/content/gopath'

r = subprocess.run(['go', 'version'], capture_output=True, text=True)
print(r.stdout.strip())

📦 Downloading Go 1.22.4…
✅ Go 1.22.4 installed
go version go1.22.4 linux/amd64


In [ ]:
import os

os.makedirs('/content/gpu-worker', exist_ok=True)

# ── main.go ───────────────────────────────────────────────────
MAIN_GO = r'''package main

import (
	"context"
	"fmt"
	"io"
	"net"
	"os"
	"os/signal"
	"strings"
	"sync"
	"syscall"
	"time"

	libp2p "github.com/libp2p/go-libp2p"
	dht "github.com/libp2p/go-libp2p-kad-dht"
	"github.com/libp2p/go-libp2p/core/network"
	"github.com/libp2p/go-libp2p/core/peer"
	drouting "github.com/libp2p/go-libp2p/p2p/discovery/routing"
	dutil "github.com/libp2p/go-libp2p/p2p/discovery/util"
	"github.com/multiformats/go-multiaddr"
)

const (
	ProtocolID    = "/gpu-rpc/1.0.0"
	RendezvousStr = "llama-gpu-pool-v1"
	RPCAddr       = "127.0.0.1:9999"
)

var bootstrapAddrs = []string{
	"/dnsaddr/bootstrap.libp2p.io/p2p/QmNnooDu7bfjPFoTZYxMNLWUQJyrVwtbZg5gBMjTezGAJN",
	"/dnsaddr/bootstrap.libp2p.io/p2p/QmQCU2EcMqAqQPR2i9bChDtGNJchTbq5TbXJJ16u19uLTa",
	"/dnsaddr/bootstrap.libp2p.io/p2p/QmbLHAnMoJPWSCR5Zhtx6BHJX9KiKNN6tpvbUcqanj75Nb",
	"/dnsaddr/bootstrap.libp2p.io/p2p/QmcZf59bWwK5XFi76CZX8cbJ4BhTzzA3gU1ZjYZcYW3dwt",
}

func parsePeers(addrs []string) []peer.AddrInfo {
	var out []peer.AddrInfo
	for _, a := range addrs {
		ma, err := multiaddr.NewMultiaddr(a)
		if err != nil {
			continue
		}
		pi, err := peer.AddrInfoFromP2pAddr(ma)
		if err != nil {
			continue
		}
		out = append(out, *pi)
	}
	return out
}

func handleStream(s network.Stream) {
	defer s.Close()
	pid := s.Conn().RemotePeer().String()
	if len(pid) > 16 {
		pid = pid[:16] + "…"
	}
	fmt.Printf("📥 Stream from %s → %s\n", pid, RPCAddr)

	conn, err := net.DialTimeout("tcp", RPCAddr, 5*time.Second)
	if err != nil {
		fmt.Printf("❌ RPC dial: %v\n", err)
		s.Reset()
		return
	}
	defer conn.Close()

	var wg sync.WaitGroup
	wg.Add(2)
	go func() { defer wg.Done(); io.Copy(conn, s) }()
	go func() { defer wg.Done(); io.Copy(s, conn) }()
	wg.Wait()
	fmt.Printf("🔌 Stream from %s closed\n", pid)
}

func main() {
	ctx, cancel := context.WithCancel(context.Background())
	defer cancel()

	bsPeers := parsePeers(bootstrapAddrs)

	fmt.Println("🚀 Starting libp2p GPU Worker…")
	h, err := libp2p.New(
		libp2p.EnableAutoRelayWithStaticRelays(bsPeers),
		libp2p.EnableHolePunching(),
		libp2p.NATPortMap(),
	)
	if err != nil {
		fmt.Fprintln(os.Stderr, "❌ host:", err)
		os.Exit(1)
	}
	defer h.Close()

	h.SetStreamHandler(ProtocolID, handleStream)

	kad, err := dht.New(ctx, h, dht.Mode(dht.ModeClient))
	if err != nil {
		fmt.Fprintln(os.Stderr, "❌ dht:", err)
		os.Exit(1)
	}

	fmt.Println("🌐 Connecting to bootstrap peers…")
	connected := 0
	for _, pi := range bsPeers {
		tctx, tc := context.WithTimeout(ctx, 10*time.Second)
		if h.Connect(tctx, pi) == nil {
			connected++
		}
		tc()
	}
	fmt.Printf("   %d/%d bootstrap peers reached\n", connected, len(bsPeers))

	_ = kad.Bootstrap(ctx)
	time.Sleep(4 * time.Second)

	disc := drouting.NewRoutingDiscovery(kad)
	dutil.Advertise(ctx, disc, RendezvousStr)
	fmt.Printf("📡 Advertising on rendezvous '%s'\n", RendezvousStr)

	printAddrs := func(label string) {
		fmt.Printf("\n%s\n", label)
		for _, a := range h.Addrs() {
			tag := "  "
			if strings.Contains(a.String(), "circuit") {
				tag = "🔗"
			}
			fmt.Printf("   %s %s/p2p/%s\n", tag, a, h.ID())
		}
	}

	fmt.Println("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
	fmt.Printf("✅ Worker ready\n🆔 Peer ID: %s\n", h.ID())
	printAddrs("📋 Initial addresses:")

	fmt.Println("\n⏳ Waiting 15s for relay addresses to appear…")
	time.Sleep(15 * time.Second)

	printAddrs("📋 Final addresses (🔗 = relay circuit, preferred for coordinator):")
	fmt.Println("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
	fmt.Println("⚠️  KEEP THIS CELL RUNNING")

	sig := make(chan os.Signal, 1)
	signal.Notify(sig, syscall.SIGINT, syscall.SIGTERM)
	<-sig
	fmt.Println("\n👋 Shutting down worker")
}
'''

# ── go.mod ────────────────────────────────────────────────────
GO_MOD = '''module gpu-worker

go 1.21

require (
	github.com/libp2p/go-libp2p v0.35.0
	github.com/libp2p/go-libp2p-kad-dht v0.25.2
	github.com/multiformats/go-multiaddr v0.13.0
)
'''

with open('/content/gpu-worker/main.go', 'w') as f:
    f.write(MAIN_GO)

with open('/content/gpu-worker/go.mod', 'w') as f:
    f.write(GO_MOD)

print("✅ Worker source written to /content/gpu-worker/")

✅ Worker source written to /content/gpu-worker/


In [ ]:
import subprocess, os, shutil

os.environ['PATH']   = '/usr/local/go/bin:' + os.environ.get('PATH', '')
os.environ['GOPATH'] = '/content/gopath'

DRIVE_BIN = '/content/drive/MyDrive/llama-p2p-worker'
LOCAL_BIN  = '/content/llama-p2p-worker'

if os.path.exists(DRIVE_BIN):
    shutil.copy(DRIVE_BIN, LOCAL_BIN)
    os.chmod(LOCAL_BIN, 0o755)
    sz = os.path.getsize(LOCAL_BIN) // (1024 * 1024)
    print(f"✅ Loaded from Drive cache ({sz} MB) — skipping build")
else:
    print("🔨 Building libp2p worker…")
    print("   First run: ~5 min (downloading go-libp2p modules)\n")

    env = os.environ.copy()

    r = subprocess.run(
        ['go', 'mod', 'tidy'],
        cwd='/content/gpu-worker', env=env,
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f"❌ go mod tidy failed:\n{r.stderr}")
        raise SystemExit(1)

    r = subprocess.run(
        ['go', 'build', '-o', LOCAL_BIN, '.'],
        cwd='/content/gpu-worker', env=env,
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f"❌ go build failed:\n{r.stderr}")
        raise SystemExit(1)

    os.chmod(LOCAL_BIN, 0o755)
    shutil.copy(LOCAL_BIN, DRIVE_BIN)
    sz = os.path.getsize(LOCAL_BIN) // (1024 * 1024)
    print(f"✅ Built ({sz} MB) and cached to Drive")

✅ Loaded from Drive cache (34 MB) — skipping build


In [ ]:
import requests, os, subprocess, shutil, glob

DRIVE_DIR = '/content/drive/MyDrive/llama-rpc-cache'
LOCAL_DIR = '/content/'

LOCAL_RPC = os.path.join(LOCAL_DIR, 'llama-rpc-server')
DRIVE_RPC = os.path.join(DRIVE_DIR, 'llama-rpc-server')

# Check cache
if os.path.exists(DRIVE_RPC) and any(glob.glob(os.path.join(DRIVE_DIR, "*.so*"))):
    for item in os.listdir(DRIVE_DIR):
        shutil.copy(os.path.join(DRIVE_DIR, item), os.path.join(LOCAL_DIR, item))
    os.chmod(LOCAL_RPC, 0o755)
    print("✅ Loaded llama-rpc-server and dynamic libraries from Drive cache")
else:
    print("🔍 Fetching latest llama.cpp release from GitHub…")
    api = requests.get("https://api.github.com/repos/ggerganov/llama.cpp/releases/latest").json()

    asset = next(
        (a for a in api['assets']
         if 'ubuntu' in a['name'].lower()
         and 'x64' in a['name'].lower()
         and not any(x in a['name'].lower() for x in ['openvino', 'vulkan', 'sycl', 'rocm'])),
        None
    )
    if not asset:
        raise RuntimeError("No standard Linux x64 binary found in latest release")

    print(f"📥 Downloading {asset['name']}…")
    with requests.get(asset['browser_download_url'], stream=True) as r:
        r.raise_for_status()
        with open('/tmp/llama.tar.gz', 'wb') as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)

    if os.path.exists('/tmp/llama_extracted'):
        shutil.rmtree('/tmp/llama_extracted')
    os.makedirs('/tmp/llama_extracted')

    subprocess.run(['tar', '-xzf', '/tmp/llama.tar.gz', '-C', '/tmp/llama_extracted'], check=True)

    # 1. Locate and copy the executable binary
    found_bin = None
    for root, _, files in os.walk('/tmp/llama_extracted'):
        for name in ['rpc-server', 'llama-rpc-server']:
            if name in files:
                found_bin = os.path.join(root, name)
                break
        if found_bin: break

    if not found_bin:
        raise RuntimeError("Neither 'rpc-server' nor 'llama-rpc-server' found in archive")

    shutil.copy(found_bin, LOCAL_RPC)
    os.chmod(LOCAL_RPC, 0o755)

    # 2. Locate and copy all essential .so files (libggml.so, llama libraries, etc.)
    libs_found = 0
    os.makedirs(DRIVE_DIR, exist_ok=True)

    for root, _, files in os.walk('/tmp/llama_extracted'):
        for file in files:
            if '.so' in file:
                src_path = os.path.join(root, file)
                shutil.copy(src_path, os.path.join(LOCAL_DIR, file))
                shutil.copy(src_path, os.path.join(DRIVE_DIR, file))
                libs_found += 1

    shutil.copy(LOCAL_RPC, DRIVE_RPC)
    print(f"✅ Downloaded executable and {libs_found} dependency libraries to Local & Drive cache")

✅ Loaded llama-rpc-server and dynamic libraries from Drive cache


In [ ]:
import subprocess, os

ok = True

r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print(f"✅ GPU    : {r.stdout.strip()}")
else:
    print("❌ GPU    : not detected"); ok = False

for label, path in [
    ('rpc-server ', '/content/llama-rpc-server'),
    ('p2p-worker ', '/content/llama-p2p-worker'),
]:
    if os.path.exists(path):
        sz = os.path.getsize(path) // (1024 * 1024)
        print(f"✅ {label}: {path}  ({sz} MB)")
    else:
        print(f"❌ {label}: NOT FOUND — rerun cell 5 or 6"); ok = False

print()
if ok:
    print("🚀 All green — run Cell 8 to launch!")
else:
    print("⚠️  Fix issues above before launching.")

✅ GPU    : Tesla T4, 15360 MiB, 14913 MiB
✅ rpc-server : /content/llama-rpc-server  (0 MB)
✅ p2p-worker : /content/llama-p2p-worker  (34 MB)

🚀 All green — run Cell 8 to launch!


In [ ]:
#DO NOT RUN

import subprocess, threading, time, os, sys

os.environ['PATH'] = '/usr/local/go/bin:' + os.environ.get('PATH', '')

def stream(proc, tag):
    for line in iter(proc.stdout.readline, b''):
        sys.stdout.write(f"[{tag}] {line.decode()}")
        sys.stdout.flush()

# ── llama-rpc-server (background) ────────────────────────────
print("🎯 Starting llama-rpc-server on 127.0.0.1:9999…")
rpc = subprocess.Popen(
    ['/content/llama-rpc-server', '-p', '9999', '--host', '127.0.0.1'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
threading.Thread(target=stream, args=(rpc, 'RPC'), daemon=True).start()
time.sleep(3)

if rpc.poll() is not None:
    raise RuntimeError("❌ llama-rpc-server failed to start — check output above")
print("✅ llama-rpc-server running\n")

# ── libp2p worker ─────────────────────────────────────────────
print("🌐 Starting libp2p worker…")
worker = subprocess.Popen(
    ['/content/llama-p2p-worker'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
threading.Thread(target=stream, args=(worker, 'P2P'), daemon=True).start()

print("\n⚠️  KEEP THIS CELL RUNNING")
print("    Watch for  ✅ Worker ready  then share the Peer ID with the coordinator\n")

# ── watchdog loop ─────────────────────────────────────────────
try:
    while True:
        time.sleep(5)
        if rpc.poll() is not None:
            print("\n❌ llama-rpc-server died — rerun this cell"); break
        if worker.poll() is not None:
            print("\n❌ libp2p worker died — rerun this cell"); break
except KeyboardInterrupt:
    print("\n🛑 Stopping…")
    rpc.terminate()
    worker.terminate()

🎯 Starting llama-rpc-server on 127.0.0.1:9999…
[RPC] load_backend: loaded RPC backend from /content/libggml-rpc.so
[RPC] load_backend: loaded CPU backend from /content/libggml-cpu-skylakex.so
✅ llama-rpc-server running

🌐 Starting libp2p worker…

⚠️  KEEP THIS CELL RUNNING
    Watch for  ✅ Worker ready  then share the Peer ID with the coordinator

[P2P] 🚀 Starting libp2p GPU Worker…
[P2P] 2026/05/20 22:32:47 failed to sufficiently increase receive buffer size (was: 208 kiB, wanted: 7168 kiB, got: 416 kiB). See https://github.com/quic-go/quic-go/wiki/UDP-Buffer-Sizes for details.
[P2P] 🌐 Connecting to bootstrap peers…
[P2P]    4/4 bootstrap peers reached
[P2P] 📡 Advertising on rendezvous 'llama-gpu-pool-v1'
[P2P] 
[P2P] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[P2P] ✅ Worker ready
[P2P] 🆔 Peer ID: 12D3KooWPxeULettPVNDS4gD37XGb3WMA3jazunQUDvjoV8AhRfW
[P2P] 
[P2P] 📋 Initial addresses:
[P2P]       /ip4/34.148.247.216/tcp/45457/p2p/12D3KooWPxeULettPVNDS4gD37XGb3WMA3jazunQUDvjoV8AhR

In [ ]:
import subprocess, threading, time, os, sys

os.environ['PATH'] = '/usr/local/go/bin:' + os.environ.get('PATH', '')

# 1. Write the custom libp2p worker source code
worker_code = """
package main

import (
	"context"
	"fmt"
	"io"
	"net"
	"time"

	libp2p "github.com/libp2p/go-libp2p"
	dht "github.com/libp2p/go-libp2p-kad-dht"
	"github.com/libp2p/go-libp2p/core/network"
	"github.com/libp2p/go-libp2p/core/peer"
	drouting "github.com/libp2p/go-libp2p/p2p/discovery/routing"
	"github.com/libp2p/go-libp2p/p2p/discovery/util"
	"github.com/multiformats/go-multiaddr"
)

const (
	ProtocolID    = "/gpu-rpc/1.0.0"
	RendezvousStr = "llama-gpu-pool-v1"
)

var bootstrapAddrs = []string{
	"/dnsaddr/bootstrap.libp2p.io/p2p/QmNnooDu7bfjPFoTZYxMNLWUQJyrVwtbZg5gBMjTezGAJN",
	"/dnsaddr/bootstrap.libp2p.io/p2p/QmQCU2EcMqAqQPR2i9bChDtGNJchTbq5TbXJJ16u19uLTa",
	"/dnsaddr/bootstrap.libp2p.io/p2p/QmbLHAnMoJPWSCR5Zhtx6BHJX9KiKNN6tpvbUcqanj75Nb",
	"/dnsaddr/bootstrap.libp2p.io/p2p/QmcZf59bWwK5XFi76CZX8cbJ4BhTzzA3gU1ZjYZcYW3dwt",
}

func main() {
	ctx := context.Background()

	var relays []peer.AddrInfo
	for _, a := range bootstrapAddrs {
		ma, _ := multiaddr.NewMultiaddr(a)
		pi, _ := peer.AddrInfoFromP2pAddr(ma)
		relays = append(relays, *pi)
	}

	h, err := libp2p.New(
		libp2p.ListenAddrStrings("/ip4/0.0.0.0/tcp/0"),
		libp2p.EnableRelay(),
		libp2p.EnableHolePunching(),
        // FORCING private reachability so it immediately grabs a relay circuit
		libp2p.ForceReachabilityPrivate(),
		libp2p.EnableAutoRelayWithStaticRelays(relays),
	)
	if err != nil {
		panic(err)
	}

	h.SetStreamHandler(ProtocolID, func(s network.Stream) {
		fmt.Println("\\n[P2P] 🔗 Incoming connection from Coordinator!")
		conn, err := net.Dial("tcp", "127.0.0.1:9999")
		if err != nil {
			fmt.Println("[P2P] ❌ Failed to connect to local RPC:", err)
			s.Reset()
			return
		}
		go io.Copy(s, conn)
		go io.Copy(conn, s)
	})

	fmt.Println("[P2P] 🌐 Connecting to bootstrap peers...")
	for _, pi := range relays {
		h.Connect(ctx, pi)
	}

	kDHT, _ := dht.New(ctx, h, dht.Mode(dht.ModeClient))
	kDHT.Bootstrap(ctx)

	fmt.Println("[P2P] ⏳ Waiting 10s to secure a public relay circuit...")
	time.Sleep(10 * time.Second)

	routingDiscovery := drouting.NewRoutingDiscovery(kDHT)
	util.Advertise(ctx, routingDiscovery, RendezvousStr)

	fmt.Println("[P2P] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
	fmt.Println("[P2P] ✅ Worker ready and advertising on DHT!")
	fmt.Printf("[P2P] 🆔 Peer ID: %s\\n", h.ID().String())
	fmt.Println("[P2P] 📋 Active Addresses:")
	for _, addr := range h.Addrs() {
		fmt.Printf("[P2P]       %s/p2p/%s\\n", addr.String(), h.ID().String())
	}
	fmt.Println("[P2P] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

	select {}
}
"""

with open("/content/worker.go", "w") as f:
    f.write(worker_code)

# 2. Compile the worker
print("⚙️ Compiling custom libp2p worker (this takes ~30 seconds once)…")
os.system("cd /content && rm -f go.mod go.sum && go mod init worker")
os.system("cd /content && go get github.com/libp2p/go-libp2p@latest github.com/libp2p/go-libp2p-kad-dht@latest github.com/multiformats/go-multiaddr@latest")
os.system("cd /content && go build -o custom-p2p-worker worker.go")

def stream(proc, tag):
    for line in iter(proc.stdout.readline, b''):
        sys.stdout.write(f"[{tag}] {line.decode()}")
        sys.stdout.flush()

# 3. Start llama-rpc-server
print("\n🎯 Starting llama-rpc-server on 127.0.0.1:9999…")
rpc = subprocess.Popen(
    ['/content/llama-rpc-server', '-p', '9999', '--host', '127.0.0.1'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
threading.Thread(target=stream, args=(rpc, 'RPC'), daemon=True).start()
time.sleep(3)

if rpc.poll() is not None:
    raise RuntimeError("❌ llama-rpc-server failed to start")

# 4. Start the custom libp2p worker
print("\n🌐 Starting custom libp2p worker…")
worker = subprocess.Popen(
    ['/content/custom-p2p-worker'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
threading.Thread(target=stream, args=(worker, 'SYS'), daemon=True).start()

print("\n⚠️  KEEP THIS CELL RUNNING")

# 5. Watchdog loop
try:
    while True:
        time.sleep(5)
        if rpc.poll() is not None:
            print("\n❌ llama-rpc-server died"); break
        if worker.poll() is not None:
            print("\n❌ p2p worker died"); break
except KeyboardInterrupt:
    print("\n🛑 Stopping…")
    rpc.terminate()
    worker.terminate()

⚙️ Compiling custom libp2p worker (this takes ~30 seconds once)…

🎯 Starting llama-rpc-server on 127.0.0.1:9999…
[RPC] load_backend: loaded RPC backend from /content/libggml-rpc.so
[RPC] load_backend: loaded CPU backend from /content/libggml-cpu-skylakex.so

🌐 Starting custom libp2p worker…

⚠️  KEEP THIS CELL RUNNING
[SYS] [P2P] 🌐 Connecting to bootstrap peers...
[SYS] 2026/05/20 23:31:40 failed to sufficiently increase receive buffer size (was: 208 kiB, wanted: 7168 kiB, got: 416 kiB). See https://github.com/quic-go/quic-go/wiki/UDP-Buffer-Sizes for details.
[SYS] [P2P] ⏳ Waiting 10s to secure a public relay circuit...
[SYS] [P2P] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[SYS] [P2P] ✅ Worker ready and advertising on DHT!
[SYS] [P2P] 🆔 Peer ID: 12D3KooWBiYE3WLitDZ9fAxhBoswv41JV2C2EATMFaD7yT26Xt7P
[SYS] [P2P] 📋 Active Addresses:
[SYS] [P2P]       /ip4/15.235.144.210/tcp/4001/p2p/QmcZf59bWwK5XFi76CZX8cbJ4BhTzzA3gU1ZjYZcYW3dwt/p2p-circuit/p2p/12D3KooWBiYE3WLitDZ9fAxhBoswv41JV2C2

## Keep-Alive Script

**Prevents Colab from disconnecting**

1. Press **F12** to open browser console
2. Paste this code:

```javascript
function KeepAlive() {
  const time = new Date().toLocaleTimeString();
  console.log(`[${time}] Keeping Colab alive...`);
  const button = document.querySelector('colab-connect-button');
  if (button) {
    const shadow = button.shadowRoot;
    const connect = shadow.querySelector('#connect');
    if (connect && connect.textContent.includes('Connect')) {
      connect.click();
      console.log('Reconnected!');
    }
  }
}

setInterval(KeepAlive, 60000);
console.log('✅ Keep-alive script started!');
```

3. Press **Enter**
4. You should see: `✅ Keep-alive script started!`